# ch03 练习：特征选择与降维

与讲解版逐 Cell 对应。核心代码被挖空，`assert` 验收**保留**——
跑通所有 `assert` 即自查通过。卡住就回讲解版看对应小节。

## 一、考点回顾

| 竞赛评分点 | 本节覆盖 |
|---|---|
| 模型训练 40% | VarianceThreshold / SelectKBest / RFE / RFECV / PCA / t-SNE |
| 模型选择 10% | 过滤 vs 包裹 vs 变换的选型 |

## 二、API 速查

| 类 | 关键参数 | 一句话 |
|---|---|---|
| `VarianceThreshold` | `threshold` | 删低方差列（怕量纲） |
| `SelectKBest` | `score_func` `k` | 统计量打分 |
| `RFE` / `RFECV` | `n_features_to_select` / `cv` `scoring` | 包裹法 |
| `PCA` | `n_components` | 方差降维（先 scale！） |
| `TSNE` | `init` `perplexity` | 只做可视化 |

In [ ]:
import warnings
from functools import partial
from pathlib import Path

import numpy as np
import pandas as pd

warnings.simplefilter("ignore")
pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 40)

import sklearn
from sklearn.decomposition import PCA
from sklearn.feature_selection import (RFE, RFECV, SelectKBest,
                                       VarianceThreshold, chi2, f_classif,
                                       mutual_info_classif)
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.manifold import TSNE
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler

DATA = Path("data")

dev = pd.read_csv(DATA / "defect_ml.csv")

NUM = ["负荷值", "温度", "湿度", "投运年限", "巡检耗时分钟"]
CAT = ["设备类型", "缺陷类型"]
TARGET = "是否危急"

X = dev[NUM + CAT]
y = dev[TARGET]

Xs_tr, Xs_te, ys_tr, ys_te = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

# 本章统一口径：训练段 fit 的 median 填充（特征选择不接受 NaN）
imp = SimpleImputer(strategy="median").fit(Xs_tr[NUM])
Xdf = pd.DataFrame(imp.transform(Xs_tr[NUM]), columns=NUM)   # DataFrame：特征名可读
Xte_df = pd.DataFrame(imp.transform(Xs_te[NUM]), columns=NUM)
Xi = Xdf.to_numpy()
ss = StandardScaler().fit(Xi)
Xs_out = ss.transform(Xi)                                    # 标准化矩阵（含负值）

print("sklearn", sklearn.__version__)
print("Xdf", Xdf.shape, "| Xs_out", Xs_out.shape, "| 正例率",
      round(float(ys_tr.mean()), 4))

In [ ]:
# 脚手架：写在 @@todo 之外，练习版原样保留
def probe(fn, *args, **kwargs):
    """把「调用可能抛异常」写成返回值，避免在挖空块里写 try/except。

    返回 ("ok", 结果) 或 ("err", 异常类型名)。
"""
    try:
        return "ok", fn(*args, **kwargs)
    except Exception as exc:
        return "err", type(exc).__name__


print("脚手架就绪：probe")

## 题 1：方差筛选与量纲陷阱（对应讲解 §3.1-3.2）

In [ ]:
# TODO(1)：定义 threshold=100 的 VarianceThreshold，fit 训练段并变换
# 提示：VarianceThreshold(threshold=...).fit(Xdf)；vt.transform(Xdf)
vt = ____
Xvt = ____
print("threshold=100 的 variances_ (真方差):", np.round(vt.variances_, 2).tolist())
print("threshold=100 保留:", vt.get_feature_names_out().tolist(),
      "| 变换后形状:", Xvt.shape)

# 版本坑（sklearn 1.9）：threshold=0 时 variances_ 被改写成 min(方差, ptp)
vt0 = VarianceThreshold(threshold=0.0).fit(Xdf)
ptp = np.ptp(Xdf.to_numpy(), axis=0)
print("threshold=0 的 variances_:", np.round(vt0.variances_, 2).tolist(),
      "← 不是方差！是 min(方差, ptp)")
print("各列 ptp:", np.round(ptp, 2).tolist())
print("min(方差, ptp) 验证:",
      bool(np.allclose(vt0.variances_, np.minimum(vt.variances_, ptp))))

# 陷阱演示：标准化后方差全为 1，方差阈值法彻底失效
vt_scaled = VarianceThreshold(threshold=0.5).fit(Xs_out)
print("scale 后每列方差全为 1 → threshold=0.5 全保留:",
      vt_scaled.get_feature_names_out().tolist())

assert np.allclose(np.round(vt.variances_, 2), [22063.89, 57.05, 129.85, 74.68, 138.52])
assert vt.get_feature_names_out().tolist() == ["负荷值", "湿度", "巡检耗时分钟"]
assert Xvt.shape == (480, 3)
assert np.allclose(vt0.variances_, [795.12, 44.4, 66.8, 29.0, 72.0])
assert vt0.get_support(indices=True).tolist() == [0, 1, 2, 3, 4]
assert len(vt_scaled.get_feature_names_out()) == 5, "scale 后方差全 1，阈值法失效"

**为什么这样做**：方差筛选的验收要同时看 `variances_` 与
`get_feature_names_out()`——留下的列是不是「量纲大」而不是「信号强」，
一眼就能对出来。

## 题 2：SelectKBest + f_classif（对应讲解 §3.3）

In [ ]:
# TODO(2)：定义 SelectKBest(f_classif, k=3)，fit 训练段并变换
# 提示：SelectKBest(f_classif, k=...)；fit 需要同时给 Xdf 与 ys_tr
skb = ____
Xk = ____
print("F 分数:", np.round(skb.scores_, 4).tolist())
print("p 值:", [f"{v:.3e}" for v in skb.pvalues_])
print("k=3 选中:", skb.get_feature_names_out().tolist())
print("support:", skb.get_support().tolist())
print("变换后形状:", Xk.shape, "← transform 只保留选中的 3 列")

assert np.allclose(np.round(skb.scores_, 4), [17.938, 37.2561, 1.0475, 0.1498, 0.092])
assert skb.get_feature_names_out().tolist() == ["负荷值", "温度", "湿度"]
assert skb.get_support().tolist() == [True, True, True, False, False]
assert Xk.shape == (480, 3)
assert float(skb.pvalues_[0]) < 1e-4 and float(skb.pvalues_[1]) < 1e-6

## 题 3：三种评分三种口味（对应讲解 §3.4）

In [ ]:
# TODO(3)：用 probe 验证：chi2 吃到标准化数据（含负值）会怎样
# 提示：先定义 SelectKBest(chi2, k=3)，再 probe(skb_neg.fit, Xs_out, ys_tr)
skb_neg = ____
status, msg = ____
print("chi2 吃到标准化数据(含负值) →", status, "|", msg)

# TODO(4)：在原始非负的 Xdf 上用 chi2 做 k=3 的特征选择
# 提示：SelectKBest(chi2, k=3).fit(Xdf, ys_tr)
skb_chi = ____
print("chi2 选中:", skb_chi.get_feature_names_out().tolist())
print("chi2 分数:", np.round(skb_chi.scores_, 2).tolist())

# TODO(5)：用 mutual_info_classif 做 k=3 的特征选择（用 partial 固定 random_state=42）
# 提示：mi_func = partial(mutual_info_classif, random_state=42) 已在 HEADER 导入
mi_func = ____
skb_mi = ____
print("mutual_info 互信息:", np.round(skb_mi.scores_, 4).tolist())
print("mutual_info 选中:", skb_mi.get_feature_names_out().tolist())
print("f_classif 选中  : 负荷值 / 温度 / 湿度")
print("mutual_info 选中: 负荷值 / 温度 / 投运年限 ← 只有它看到了投运年限的非线性信号")

assert status == "err" and msg == "ValueError", "chi2 要求非负"
assert skb_chi.get_feature_names_out().tolist() == ["负荷值", "温度", "湿度"]
assert round(float(skb_chi.scores_[0]), 2) == 905.5
assert skb_mi.get_feature_names_out().tolist() == ["负荷值", "温度", "投运年限"]
assert round(float(skb_mi.scores_[1]), 4) == 0.0555

**为什么这样做**：chi2 的非负约束用 probe 验证而不是背结论；
mutual_info 用 `partial` 固定种子。两个选中集只差一列——
投运年限的非线性信号只有互信息看得见。

## 题 4：RFE 与 RFECV（对应讲解 §3.5）

In [ ]:
# TODO(6)：定义 RFE(LogisticRegression(max_iter=1000), n_features_to_select=3) 并 fit
# 提示：RFE(模型, n_features_to_select=...)；fit 要给 Xdf 与 ys_tr
lr = ____
rfe = ____
print("RFE 选中:", rfe.get_feature_names_out().tolist())
print("RFE ranking:", rfe.ranking_.tolist(), "← 1 是保留，2/3 是第几轮被淘汰")

# TODO(7)：定义 RFECV：cv 用分层 5 折（shuffle=True, 种子 42），scoring 用 accuracy
# 提示：RFECV(模型, step=1, cv=StratifiedKFold(...), scoring=..., min_features_to_select=1, n_jobs=1)
rfecv = ____
print("RFECV 最优特征数:", int(rfecv.n_features_))
print("RFECV 选中:", rfecv.get_feature_names_out().tolist())
print("各特征数下的 CV 均值:", np.round(rfecv.cv_results_["mean_test_score"], 4).tolist())

assert rfe.get_feature_names_out().tolist() == ["温度", "湿度", "巡检耗时分钟"]
assert rfe.ranking_.tolist() == [3, 1, 1, 2, 1]
assert int(rfecv.n_features_) == 5, "CV 说 5 个全要：删谁降谁"
assert rfecv.get_feature_names_out().tolist() == NUM
assert np.allclose(np.round(rfecv.cv_results_["mean_test_score"], 4),
                   [0.7833, 0.7875, 0.7875, 0.7979, 0.8083])

## 题 5：PCA 的绑架与降维（对应讲解 §3.6）

In [ ]:
# TODO(8)：对未标准化的 Xi fit 全量 PCA，看第一主成分吃掉多少方差
# 提示：PCA(random_state=42).fit(Xi)
pca_raw = ____
ratio_raw = ____
print("未 scale 的方差贡献率:", np.round(ratio_raw, 4).tolist(),
      "← 负荷值一列吃掉 98%")

# TODO(9)：对标准化后的 Xs_out 定义 PCA(n_components=2)，transform 得到 Z
# 提示：先 PCA(random_state=42).fit(Xs_out) 看 5 个主成分的方差贡献率，再降 2 维
pca_s = ____
ratio = ____
p2 = ____
Z = ____
print("scale 后的方差贡献率:", np.round(ratio, 4).tolist())
print("累积:", np.round(np.cumsum(ratio), 4).tolist())
print("降维后形状:", Z.shape, "| 保留方差:", round(float(p2.explained_variance_ratio_.sum()), 4))
print("PC1 载荷:", np.round(p2.components_[0], 4).tolist())
top = NUM[int(np.argmax(np.abs(p2.components_[0])))]
print("PC1 载荷绝对值最大的特征:", top)

# TODO(10)：用 inverse_transform 重构并计算重构 MSE
# 提示：rec = p2.inverse_transform(Z)；err = float(np.mean((rec - Xs_out) ** 2))
rec = ____
err = ____
print("逆变换重构 MSE:", round(err, 4), "← 57% 的方差信息被丢掉了")

assert round(float(ratio_raw[0]), 4) == 0.9822
assert np.allclose(np.round(ratio, 4), [0.214, 0.2127, 0.2003, 0.1977, 0.1753])
assert Z.shape == (480, 2) and round(float(p2.explained_variance_ratio_.sum()), 4) == 0.4267
assert top == "负荷值"
assert round(err, 4) == 0.5733

## 题 6：t-SNE 与三条纪律（对应讲解 §3.7）

In [ ]:
# TODO(11)：定义 TSNE(n_components=2, random_state=42, init="pca") 并 fit_transform
# 提示：ts.fit_transform(Xs_out)；结果存入 Zt
ts = ____
Zt = ____
print("t-SNE 输出形状:", Zt.shape)
print("KL 散度:", round(float(ts.kl_divergence_), 4), "(越小局部结构保持越好)")
print("迭代次数:", int(ts.n_iter_))

# 三条纪律：不可 transform 新数据 / 不进 Pipeline / 结果不可复现到小数点
# 注意：TSNE 没有 transform 属性，ts.transform 在求值时就会 AttributeError，
# 所以用 getattr 把「属性访问」本身交给 probe 接住
st = probe(getattr, ts, "transform")
print("访问 ts.transform →", st[0], "|", st[1])

assert Zt.shape == (480, 2)
assert float(ts.kl_divergence_) > 0
assert st[0] == "err" and st[1] == "AttributeError"

---

## 综合自查

1. 题 1 里 threshold=100 留下的三列，是「信号强」还是「量纲大」？
2. 题 3 两个选中集只差一列，差的那列说明什么？
3. 题 4 的 RFECV 给出「5 个全要」，如果评委问你为什么还要做特征选择，怎么答？
4. 题 5 的 0.9822 和 0.4267 分别是什么？哪个才是 PCA 该有的样子？
5. t-SNE 的输出能塞进 Pipeline 当特征吗？为什么？

全部答得上来，进入 ch04（回归模型）。